In [4]:
import duckdb
import pandas as pd
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor

In [5]:
con = duckdb.connect()

FEATURE_PATH = "data/processed/CA_1_features_v1.parquet"

df = con.execute(f"""
SELECT
    item_id,
    dept_id,
    cat_id,
    store_id,
    state_id,
    date,
    units_sold,
    sell_price,
    weekday,
    month,
    year,
    event_name_1,
    event_type_1,
    snap_CA,
    lag_1,
    lag_7,
    lag_28,
    rolling_mean_7,
    rolling_mean_28
FROM read_parquet('{FEATURE_PATH}')
WHERE
    is_available = 1
    AND lag_28 IS NOT NULL
""").fetchdf()

In [6]:
max_date = df["date"].max()

test_start = max_date - pd.Timedelta(days=27)
val_start = test_start - pd.Timedelta(days=28)

train = df[df["date"] < val_start].copy()

val = df[
    (df["date"] >= val_start) &
    (df["date"] < test_start)
].copy()

test = df[df["date"] >= test_start].copy()

In [14]:
# Lag-1
mae_1 = mean_absolute_error(
    val["units_sold"],
    val["lag_1"]
)

rmse_1 = np.sqrt(
    mean_squared_error(
        val["units_sold"],
        val["lag_1"]
    )
)

# Lag-7
mae_7 = mean_absolute_error(
    val["units_sold"],
    val["lag_7"]
)

rmse_7 = np.sqrt(
    mean_squared_error(
        val["units_sold"],
        val["lag_7"]
    )
)

# Rolling mean 28
mae_roll = mean_absolute_error(
    val["units_sold"],
    val["rolling_mean_28"]
)

rmse_roll = np.sqrt(
    mean_squared_error(
        val["units_sold"],
        val["rolling_mean_28"]
    )
)

In [17]:
from catboost import CatBoostRegressor

model = CatBoostRegressor()
model.load_model("models/catboost_ca1.cbm")

CatBoostRegressor(depth=8, eval_metric='RMSE', iterations=500, learning_rate=0.05, loss_function='RMSE', od_type='Iter', od_wait=50, random_seed=42, verbose=100)

In [19]:
features = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "sell_price",
    "weekday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "snap_CA",
    "lag_1",
    "lag_7",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28"
]

cat_features = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "weekday",
    "event_name_1",
    "event_type_1"
]

for col in cat_features:
    val[col] = val[col].fillna("None").astype(str)

In [20]:
pred = model.predict(val[features])

cat_mae = mean_absolute_error(
    val["units_sold"],
    pred
)

cat_rmse = np.sqrt(
    mean_squared_error(
        val["units_sold"],
        pred
    )
)

In [21]:
comparison = pd.DataFrame({
    "model": [
        "Lag 1",
        "Lag 7",
        "Rolling Mean 28",
        "CatBoost"
    ],
    "MAE": [
        mae_1,
        mae_7,
        mae_roll,
        cat_mae
    ],
    "RMSE": [
        rmse_1,
        rmse_7,
        rmse_roll,
        cat_rmse
    ]
})

comparison.sort_values("RMSE")

,model,MAE,RMSE
3,CatBoost,1.038457,2.012824
2,Rolling Mean 28,1.087552,2.193424
0,Lag 1,1.281732,2.768171
1,Lag 7,1.309680,2.806559


In [22]:
val_results = val[
    ["item_id", "dept_id", "cat_id", "date", "units_sold"]
].copy()

val_results["prediction"] = pred
val_results["absolute_error"] = (
    val_results["units_sold"] - val_results["prediction"]
).abs()

In [23]:
val_results.sort_values(
    "absolute_error",
    ascending=False
).head(20)

,item_id,dept_id,cat_id,date,units_sold,prediction,absolute_error
4351985,HOUSEHOLD_1_465,HOUSEHOLD_1,HOUSEHOLD,2016-04-04,82,13.115444,68.884556
3909667,FOODS_3_120,FOODS_3,FOODS,2016-04-10,0,65.106962,65.106962
1997380,FOODS_3_498,FOODS_3,FOODS,2016-03-31,64,1.329039,62.670961
3739526,FOODS_3_295,FOODS_3,FOODS,2016-03-29,78,22.891713,55.108287
2651501,FOODS_3_501,FOODS_3,FOODS,2016-03-29,66,17.455655,48.544345
3909660,FOODS_3_120,FOODS_3,FOODS,2016-04-03,126,77.794855,48.205145
2552953,HOBBIES_1_278,HOBBIES_1,HOBBIES,2016-04-08,48,0.565286,47.434714
1966100,HOBBIES_1_261,HOBBIES_1,HOBBIES,2016-04-11,50,5.927372,44.072628
3909665,FOODS_3_120,FOODS_3,FOODS,2016-04-08,58,14.024744,43.975256
3909659,FOODS_3_120,FOODS_3,FOODS,2016-04-02,109,65.810833,43.189167


In [24]:
val_results.groupby("cat_id").agg(
    MAE=("absolute_error", "mean")
).sort_values("MAE", ascending=False)

,MAE
cat_id,
FOODS,1.298329
HOBBIES,0.890354
HOUSEHOLD,0.761707


In [25]:
importance = pd.DataFrame({
    "feature": features,
    "importance": model.get_feature_importance()
}).sort_values("importance", ascending=False)

importance

,feature,importance
15,rolling_mean_7,37.249885
16,rolling_mean_28,19.432287
12,lag_1,16.713716
6,weekday,9.268159
13,lag_7,4.501403
14,lag_28,4.134492
5,sell_price,2.763078
0,item_id,1.381718
9,event_name_1,1.164371
7,month,1.076189
